# Modeling

# 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

# 2. Preprocessed Data Import

In [ ]:
df_preprocessed = pd.read_csv(filepath_or_buffer='../data/preprocessed/dataset_preprocessed.csv')
df_preprocessed.head()

# 3. Data Splitting

In [ ]:
X = df_preprocessed.drop(columns=['tiempo_resolución_con_hora','tiempo_resolución'])
y = df_preprocessed['tiempo_resolución_con_hora']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Encoding

## 4.1 Encoding "Tema"

### 4.1.1 Obtain 80% of topics

In [ ]:
X_train.info()

In [ ]:
topic_counts = X_train['tema_solicitud'].value_counts(normalize=True)
topic_counts

In [ ]:
cumulative_frequency = topic_counts.drop(labels='OTRO').cumsum()
cumulative_frequency

In [ ]:
top_categories = cumulative_frequency.where(cumulative_frequency <= 0.8).dropna().index.to_list()
top_categories

### 4.1.2 Create Compact Topic Column

In [ ]:
X_train['tema_solicitud_compact'] = X_train['tema_solicitud'].where(X_train['tema_solicitud'].isin(top_categories),other='OTRO')
print(X_train['tema_solicitud_compact'].value_counts(normalize=True).cumsum())

In [ ]:
X_test['tema_solicitud_compact'] = X_test['tema_solicitud'].where(X_test['tema_solicitud'].isin(top_categories),other='OTRO')

### 4.1.3 Drop Column

In [ ]:
X_train.drop(columns=['tema_solicitud'],inplace=True)
X_train

In [ ]:
X_test.drop(columns=['tema_solicitud'],inplace=True)
X_test

## 4.2 Encode Categorical Columns

In [ ]:
categorical_columns = ['tipo_de_entrada','sexo','alcaldia_solicitud','dia_solicitud','mes_solicitud','tema_solicitud_compact']
ohe = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
for param in categorical_columns:
    param_encoded = ohe.fit_transform(X_train[[param]])
    param_columns = ohe.get_feature_names_out([param])
    X_train[param_columns] = param_encoded
    X_train.drop(columns=[param],inplace=True)
    param_encoded_test = ohe.transform(X_test[[param]])
    X_test[param_columns] = param_encoded_test
    X_test.drop(columns=[param],inplace=True)
X_train

In [ ]:
X_train.info()

# 5. Transformations

## 5.1 X Scaling

In [ ]:
sc = StandardScaler()
train_hora_sc = sc.fit_transform(X_train[['hora_sin','hora_cos']])
X_train[['hora_sin','hora_cos']] = train_hora_sc
test_hora_sc = sc.transform(X_test[['hora_sin','hora_cos']])
X_test[['hora_sin','hora_cos']] = test_hora_sc

## 5.2 Y Transform

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

# 6. Feature Importance Validation (Random Forest)

In [ ]:
X_train.info()

In [ ]:
rf_raw_target = RandomForestRegressor(n_estimators=100, max_depth=15, max_features='sqrt', n_jobs=-1, random_state=0)
rf_raw_target.fit(X_train, y_train)
importances_raw = pd.Series(rf_raw_target.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances_raw.head(20)

In [ ]:
rf_log_target = RandomForestRegressor(n_estimators=100, max_depth=15, max_features='sqrt', n_jobs=-1, random_state=0)
rf_log_target.fit(X_train, y_train_log)
importances_log = pd.Series(rf_log_target.feature_importances_, index=X_train.columns).sort_values(ascending=False)
importances_log.head(20)

In [ ]:
def original_feature(column_name):
    if column_name in ('hora_sin', 'hora_cos'):
        return 'hora_solicitud'
    else:
        for variable in categorical_columns:
            if column_name.startswith(variable):
                return variable
        return column_name

In [ ]:
importances_log.index = importances_log.index.map(original_feature)

In [ ]:
importances_log.groupby(importances_log.index).sum().sort_values(ascending=False)

# 7. Linear Regression

## 7.1 All Features

In [ ]:
X_train_1 = X_train.copy()
X_test_1 = X_test.copy()

In [ ]:
lr_1 = LinearRegression(n_jobs=-1)
model_1 = lr_1.fit(X=X_train_1, y=y_train_log)
y_pred_1 = np.expm1(model_1.predict(X_test_1))
rmse_1 = root_mean_squared_error(y_true=y_test, y_pred=y_pred_1)
mae_1 = mean_absolute_error(y_true=y_test, y_pred=y_pred_1)
r2_1 = r2_score(y_true=y_test, y_pred=y_pred_1)
print(f'RMSE 1: {rmse_1}, MAE 1: {mae_1},  R2 1: {r2_1}')

## 7.2 Without 'sexo' and 'dia'

In [ ]:
columns_to_drop_v1 = []
for column in X_train.columns:
    if column.startswith('sexo'):
        columns_to_drop_v1.append(column)
    elif column.startswith('dia'):
        columns_to_drop_v1.append(column)
print(columns_to_drop_v1)

In [ ]:
X_train_2 = X_train.drop(columns=columns_to_drop_v1)
X_test_2 = X_test.drop(columns=columns_to_drop_v1)

In [ ]:
lr_2 = LinearRegression(n_jobs=-1)
model_2 = lr_2.fit(X=X_train_2, y=y_train_log)
y_pred_2 = np.expm1(model_2.predict(X_test_2))
rmse_2 = root_mean_squared_error(y_true=y_test, y_pred=y_pred_2)
mae_2 = mean_absolute_error(y_true=y_test, y_pred=y_pred_2)
r2_2 = r2_score(y_true=y_test, y_pred=y_pred_2)
print(f'RMSE 2: {rmse_2}, MAE 2: {mae_2}, R2 2: {r2_2}')

## 7.3 Without 'sexo', 'dia', 'mes' and 'hora'

In [ ]:
columns_to_drop_v2 = []
for column in X_train.columns:
    if column.startswith('sexo'):
        columns_to_drop_v2.append(column)
    elif column.startswith('dia'):
        columns_to_drop_v2.append(column)
    elif column.startswith('mes'):
        columns_to_drop_v2.append(column)
    elif column.startswith('hora'):
        columns_to_drop_v2.append(column)
print(columns_to_drop_v2)

In [ ]:
X_train_3 = X_train.drop(columns=columns_to_drop_v2)
X_test_3 = X_test.drop(columns=columns_to_drop_v2)

In [ ]:
lr_3 = LinearRegression(n_jobs=-1)
model_3 = lr_3.fit(X=X_train_3, y=y_train_log)
y_pred_3 = np.expm1(model_3.predict(X_test_3))
rmse_3 = root_mean_squared_error(y_true=y_test, y_pred=y_pred_3)
mae_3 = mean_absolute_error(y_true=y_test, y_pred=y_pred_3)
r2_3 = r2_score(y_true=y_test, y_pred=y_pred_3)
print(f'RMSE 3: {rmse_3}, MAE 3: {mae_3}, R2 3: {r2_3}')

# 8. Random Forest Regressor

In [ ]:
columns_to_drop_final = []
for column in X_train.columns:
    if column.startswith('sexo'):
        columns_to_drop_final.append(column)
    elif column.startswith('dia'):
        columns_to_drop_final.append(column)
print(columns_to_drop_final)

In [ ]:
X_train_final = X_train.drop(columns=columns_to_drop_final)
X_test_final = X_test.drop(columns=columns_to_drop_final)

In [ ]:
param_dict = {
    'n_estimators' : [100, 200, 300, 500],
    'max_depth' : [10, 15, 20, 25],
    'min_samples_split' : [2, 5, 10, 20],
    'min_samples_leaf' : [1, 2, 5, 10]
}

In [ ]:
rfr = RandomForestRegressor(max_features='sqrt', n_jobs=-1, random_state=0)
random_search = RandomizedSearchCV(
    estimator=rfr,
    param_distributions=param_dict,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    random_state=0,
    verbose=2
)

random_search.fit(X=X_train_final, y=y_train_log)

print(random_search.best_params_)
print(random_search.best_score_)

In [ ]:
y_pred_rf = np.expm1(random_search.best_estimator_.predict(X_test_final))
rmse_rfr = root_mean_squared_error(y_true=y_test, y_pred=y_pred_rf)
mae_rfr = mean_absolute_error(y_true=y_test, y_pred=y_pred_rf)
r2_rfr = r2_score(y_true=y_test, y_pred=y_pred_rf)
print(f'RMSE RFR: {rmse_rfr}, MAE RFR: {mae_rfr}, R2 RFR: {r2_rfr}')

In [ ]:
import plotly.express as px

rf_residuals = y_test - y_pred_rf

fig = px.histogram(rf_residuals, nbins=100, title='Residuals Distribution — Random Forest')
fig.update_layout(xaxis_title='Residual (y_test - y_pred)', yaxis_title='Frequency')
fig.show()

# 9. XGBoost